<a href="https://colab.research.google.com/github/Parth-Thombre/ContextaAI/blob/main/Camera_Code_Action_Instructor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Camera, Code, Action!
### A GDSC Computer Vision Workshop — draw in the air with MediaPipe

In this notebook you'll teach a computer to **see your hand** and turn a finger into a paintbrush.

We start from a short prerecorded video of a hand moving through the air. Google's **MediaPipe Hand Landmarker** finds 21 key points on the hand in each frame. We follow one of them (the index fingertip), and an **AirCanvas** draws its path.

**What you'll produce**

| Output | What it shows |
|---|---|
| 🅰️ Hand Tracking video | The original video with all 21 landmarks, the hand skeleton, and the tracked fingertip highlighted |
| 🅱️ AirCanvas video | A clean canvas showing the drawing appear stroke by stroke |
| 🆎 Combined video | Both side by side |

**How to run it:** the whole notebook runs in the cloud on a normal CPU runtime. Run the cells from top to bottom (`Shift + Enter`), or use **Runtime → Run all**.

| # | Section |
|---|---|
| 1 | Introduction (you are here) |
| 2 | Setup |
| 3 | Load the Video |
| 4 | Understanding MediaPipe |
| 5 | Initialize HandLandmarker |
| 6 | Detect Hand Landmarks |
| 7 | Extract Finger Coordinates |
| 8 | AirCanvas |
| 9 | Generate Results |
| 10 | Experimentation |
| 11 | Optional Challenges |

---
## 2. Setup

We install MediaPipe and download the pretrained hand landmark model (~7.5 MB).

> **Why `--no-deps`?** By default MediaPipe would pull in its own copy of OpenCV, which can clash with the OpenCV that Colab already ships and may force a runtime restart. Colab already has OpenCV, NumPy and Matplotlib, so we install MediaPipe by itself plus the few small packages it still needs.

In [ ]:
!pip install -q --no-deps mediapipe==1.1.1
!pip install -q "absl-py>=2.3" certifi "flatbuffers~=25.9" "sounddevice~=0.5"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
mediapipe 1.1.1 requires sounddevice~=0.5, which is not installed.


In [ ]:
import base64
import os
import shutil
import subprocess
import time
import urllib.request

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
from IPython.display import HTML, display
from mediapipe.tasks import python as mp_tasks
from mediapipe.tasks.python import vision

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

MODEL_URL = ("https://storage.googleapis.com/mediapipe-models/hand_landmarker/"
             "hand_landmarker/float16/1/hand_landmarker.task")
MODEL_PATH = "hand_landmarker.task"
OUTPUT_DIR = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

if not os.path.exists(MODEL_PATH):
    print("Downloading hand landmark model...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)

print(f"MediaPipe {mp.__version__} | OpenCV {cv2.__version__} | NumPy {np.__version__}")
print(f"Model: {MODEL_PATH} ({os.path.getsize(MODEL_PATH) / 1e6:.1f} MB)")
print(f"FFmpeg available: {shutil.which('ffmpeg') is not None} | Running in Colab: {IN_COLAB}")

### Helper utilities

These handle the plumbing (video encoding, showing videos, and messages). You don't need to read them closely.

OpenCV writes MP4 files with a codec most browsers can't play, so `Mp4Writer` writes a temporary file and then uses **FFmpeg** to convert it to **H.264**. Colab's video player can play H.264.

In [ ]:
def to_browser_mp4(src, dst, size=None):
    """Re-encode any video to browser-friendly H.264 MP4 (optionally resized to size=(w, h))."""
    if shutil.which("ffmpeg") is None:
        shutil.copy(src, dst)
        warn("FFmpeg not found: the video may not play in the browser.")
        return dst
    vf = f"scale={size[0]}:{size[1]}" if size else "scale=trunc(iw/2)*2:trunc(ih/2)*2"
    cmd = ["ffmpeg", "-y", "-loglevel", "error", "-i", src, "-vf", vf,
           "-c:v", "libx264", "-preset", "veryfast", "-crf", "23",
           "-pix_fmt", "yuv420p", "-movflags", "+faststart", "-an", dst]
    subprocess.run(cmd, check=True)
    return dst


class Mp4Writer:
    """Collect BGR frames and save them as a browser-playable H.264 MP4."""

    def __init__(self, path, fps, size):
        self.path = path
        self.raw_path = path.replace(".mp4", "_raw.mp4")
        self.writer = cv2.VideoWriter(self.raw_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, size)
        if not self.writer.isOpened():
            raise RuntimeError(f"Could not open a video writer for {path}")

    def write(self, frame):
        self.writer.write(frame)

    def close(self):
        self.writer.release()
        to_browser_mp4(self.raw_path, self.path)
        os.remove(self.raw_path)
        return self.path


def show_video(path, width=640):
    """Embed an MP4 in the notebook output."""
    data = base64.b64encode(open(path, "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls loop autoplay muted playsinline '
                 f'src="data:video/mp4;base64,{data}"></video>'))


def download_link(path):
    """An HTML link that downloads the file straight from the notebook output."""
    data = base64.b64encode(open(path, "rb").read()).decode()
    name = os.path.basename(path)
    size_mb = os.path.getsize(path) / 1e6
    return f'<a download="{name}" href="data:video/mp4;base64,{data}">⬇️ {name}</a> ({size_mb:.1f} MB)'


def show_image(image_bgr, title=None, width=8):
    """Show an OpenCV (BGR) image with Matplotlib."""
    h, w = image_bgr.shape[:2]
    plt.figure(figsize=(width, width * h / w))
    plt.imshow(cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    if title:
        plt.title(title)
    plt.show()


def warn(message):
    display(HTML(f'<div style="padding:10px;border-left:6px solid #f4b400;background:#fff8e1;'
                 f'color:#5f4b00;font-weight:600">⚠️ {message}</div>'))


print("Helpers ready ✅")

---
## 3. Load the Video

We need a short MP4 (about 10–15 seconds) of **one hand** tracing a shape in the air, such as a star, a heart, or your initials.

**Tips for a good recording**
- Keep the whole hand in view, with the index finger clearly pointing.
- Use good lighting and a plain background if you can.
- Move steadily; very fast motion blurs the finger.

**Where the video comes from** (the first option that works is used):
1. `SAMPLE_VIDEO_URL`: a public link to an MP4. *(Instructors: paste the workshop sample video link here.)*
2. Otherwise, an **Upload** button appears below the cell. Pick an `.mp4` from your computer.

Once loaded, the video is saved as `input_video.mp4`. Re-running the cell reuses it. Set `USE_NEW_VIDEO = True` to replace it.

In [ ]:
SAMPLE_VIDEO_URL = ""        # e.g. "https://.../hand_drawing_sample.mp4" (leave empty to upload)
VIDEO_PATH = "input_video.mp4"
USE_NEW_VIDEO = False        # True = fetch/upload again even if input_video.mp4 exists
MAX_SIDE = 640               # longest side (pixels) we process at: keeps things fast on CPU


def get_input_video():
    if os.path.exists(VIDEO_PATH) and not USE_NEW_VIDEO:
        print(f"Using existing {VIDEO_PATH}")
        return VIDEO_PATH
    if SAMPLE_VIDEO_URL:
        print("Downloading sample video...")
        urllib.request.urlretrieve(SAMPLE_VIDEO_URL, VIDEO_PATH)
        return VIDEO_PATH
    if IN_COLAB:
        print("Choose an MP4 video of a hand drawing in the air:")
        uploaded = colab_files.upload()
        if not uploaded:
            raise FileNotFoundError("No file was uploaded. Run this cell again and pick an MP4.")
        name = next(iter(uploaded))
        if not name.lower().endswith((".mp4", ".mov", ".m4v")):
            warn(f"'{name}' is not an .mp4 file. We'll try to read it anyway.")
        os.replace(name, VIDEO_PATH)
        return VIDEO_PATH
    raise FileNotFoundError(f"No video found. Put an MP4 at '{VIDEO_PATH}' or set SAMPLE_VIDEO_URL.")


def compute_output_size(width, height, max_side=MAX_SIDE):
    """Shrink (never enlarge) so the longest side <= max_side, keep the aspect ratio, use even sizes."""
    scale = min(1.0, max_side / max(width, height))
    return (int(width * scale) // 2 * 2, int(height * scale) // 2 * 2)


def load_video_info(path):
    """Open the video, check that it is readable, and return its basic properties."""
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        raise FileNotFoundError(f"'{path}' is missing or empty.")
    cap = cv2.VideoCapture(path)
    ok, frame = cap.read()
    if not cap.isOpened() or not ok:
        cap.release()
        raise ValueError(f"OpenCV can't read '{path}'. Is it a valid MP4 video?")
    fps = cap.get(cv2.CAP_PROP_FPS)
    if not fps or np.isnan(fps) or fps > 240:
        fps = 30.0
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    height, width = frame.shape[:2]   # from the decoded frame, so phone rotation is respected
    return {"fps": fps, "frame_count": frame_count, "width": width, "height": height,
            "duration": frame_count / fps if frame_count > 0 else float("nan")}


video_path = get_input_video()
info = load_video_info(video_path)
FPS = info["fps"]
FRAME_SIZE = compute_output_size(info["width"], info["height"])   # (width, height)
WIDTH, HEIGHT = FRAME_SIZE

print(f"Original: {info['width']}x{info['height']} @ {FPS:.1f} fps, "
      f"{info['frame_count']} frames (~{info['duration']:.1f} s)")
print(f"Processing at: {WIDTH}x{HEIGHT}")
if info["duration"] > 30:
    warn("This video is longer than 30 s. Processing will be slow; a 10–15 s clip works best.")

preview_path = to_browser_mp4(video_path, os.path.join(OUTPUT_DIR, "input_preview.mp4"), size=FRAME_SIZE)
show_video(preview_path, width=min(WIDTH, 480))

---
## 4. Understanding MediaPipe

**MediaPipe** is Google's open-source framework for on-device machine learning. Its **Hand Landmarker** runs in two stages:

1. A **palm detector** finds where a hand is in the image.
2. A **landmark model** looks closely at that region and predicts **21 landmarks**, the joints and tips of the hand.

![Hand landmarks](https://ai.google.dev/static/mediapipe/images/solutions/hand-landmarks.png)

Each landmark has:
- `x`, `y`: position **normalized** to `[0, 1]` relative to the image width and height. (0, 0) is the top-left corner.
- `z`: depth relative to the wrist. Smaller values are closer to the camera.

Because coordinates are normalized, we multiply by the frame size to get pixels: `px = x * width`, `py = y * height`.

The important IDs for us are the fingertips: **4** (thumb), **8** (index), **12** (middle), **16** (ring) and **20** (pinky).

MediaPipe also tells us which landmarks are **connected** to form the hand skeleton. We use that list to draw the bones.

In [ ]:
LANDMARK_NAMES = [
    "WRIST",
    "THUMB_CMC", "THUMB_MCP", "THUMB_IP", "THUMB_TIP",
    "INDEX_MCP", "INDEX_PIP", "INDEX_DIP", "INDEX_TIP",
    "MIDDLE_MCP", "MIDDLE_PIP", "MIDDLE_DIP", "MIDDLE_TIP",
    "RING_MCP", "RING_PIP", "RING_DIP", "RING_TIP",
    "PINKY_MCP", "PINKY_PIP", "PINKY_DIP", "PINKY_TIP",
]

# The official hand skeleton topology, straight from MediaPipe
HAND_CONNECTIONS = [(c.start, c.end) for c in vision.HandLandmarksConnections.HAND_CONNECTIONS]

for i, name in enumerate(LANDMARK_NAMES):
    print(f"{i:2d}  {name}")
print(f"\n{len(HAND_CONNECTIONS)} connections, e.g. {HAND_CONNECTIONS[:5]}")

---
## 5. Initialize HandLandmarker

A MediaPipe task is created from **options**:

| Option | Meaning |
|---|---|
| `model_asset_path` | The `.task` model file we downloaded |
| `running_mode` | `IMAGE` (single photos), `VIDEO` (video frames, uses tracking between frames), or `LIVE_STREAM` (webcam, asynchronous) |
| `num_hands` | Maximum number of hands to find |
| `min_hand_detection_confidence` | How sure the palm detector must be to report a new hand |
| `min_hand_presence_confidence` | How sure the landmark model must be that a hand is still there |
| `min_tracking_confidence` | How well the hand must match the previous frame to keep **tracking** instead of re-detecting |

We use **`VIDEO` mode**. MediaPipe can reuse the previous frame's hand position, which is faster and smoother than detecting every frame from scratch. In this mode every frame needs a **timestamp** that always increases.

Because of those timestamps, **each video (or each fresh run) needs a new landmarker**. That's why we wrap creation in a function and open it with `with ...:`, which also closes it cleanly afterwards.

> MediaPipe prints some log lines, such as `Feedback manager requires a model with a single signature...` or `Created TensorFlow Lite XNNPACK delegate`. They are harmless and can be ignored.

In [ ]:
NUM_HANDS = 1
MIN_HAND_DETECTION_CONFIDENCE = 0.5
MIN_HAND_PRESENCE_CONFIDENCE = 0.5
MIN_TRACKING_CONFIDENCE = 0.5


def create_landmarker(num_hands, min_detection, min_presence, min_tracking):
    options = vision.HandLandmarkerOptions(
        base_options=mp_tasks.BaseOptions(model_asset_path=MODEL_PATH),
        running_mode=vision.RunningMode.VIDEO,
        num_hands=num_hands,
        min_hand_detection_confidence=min_detection,
        min_hand_presence_confidence=min_presence,
        min_tracking_confidence=min_tracking,
    )
    return vision.HandLandmarker.create_from_options(options)


# Quick check that the model loads
with create_landmarker(NUM_HANDS, MIN_HAND_DETECTION_CONFIDENCE,
                       MIN_HAND_PRESENCE_CONFIDENCE, MIN_TRACKING_CONFIDENCE) as landmarker:
    print("HandLandmarker ready ✅", type(landmarker).__name__)

---
## 6. Detect Hand Landmarks

### 6a. Try one frame first

For each frame we:
1. Convert OpenCV's **BGR** colour order to the **RGB** MediaPipe expects.
2. Wrap the pixels in an `mp.Image`.
3. Call `detect_for_video(image, timestamp_ms)`.

The result has `hand_landmarks` (a list with one entry per detected hand, each holding 21 landmarks) and `handedness` (Left or Right).

In [ ]:
def read_frame(path, index, size):
    """Read a single frame (resized) from a video."""
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, index)
    ok, frame = cap.read()
    cap.release()
    if not ok:
        raise ValueError(f"Could not read frame {index}")
    return cv2.resize(frame, size, interpolation=cv2.INTER_AREA)


def draw_hand(frame, hand_landmarks, highlight_id=None, highlight_color=(0, 255, 0)):
    """Draw the skeleton, all 21 landmarks, and (optionally) one highlighted landmark."""
    h, w = frame.shape[:2]
    points = [(int(lm.x * w), int(lm.y * h)) for lm in hand_landmarks]
    for start, end in HAND_CONNECTIONS:
        cv2.line(frame, points[start], points[end], (255, 255, 255), 2, cv2.LINE_AA)
    for p in points:
        cv2.circle(frame, p, 4, (0, 0, 255), -1, cv2.LINE_AA)
    if highlight_id is not None:
        cv2.circle(frame, points[highlight_id], 10, highlight_color, 3, cv2.LINE_AA)
    return frame


test_index = max(info["frame_count"] // 2, 0)
frame = read_frame(video_path, test_index, FRAME_SIZE)

rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
with create_landmarker(NUM_HANDS, MIN_HAND_DETECTION_CONFIDENCE,
                       MIN_HAND_PRESENCE_CONFIDENCE, MIN_TRACKING_CONFIDENCE) as landmarker:
    result = landmarker.detect_for_video(mp_image, 0)

print(f"Hands found in frame {test_index}: {len(result.hand_landmarks)}")
if result.hand_landmarks:
    hand = result.hand_landmarks[0]
    print("Handedness:", result.handedness[0][0].category_name,
          f"(score {result.handedness[0][0].score:.2f})")
    print("Index fingertip (normalized):", f"x={hand[8].x:.3f}, y={hand[8].y:.3f}, z={hand[8].z:.3f}")
    show_image(draw_hand(frame.copy(), hand, highlight_id=8), f"Frame {test_index}")
else:
    warn("No hand in this frame. That's OK; the full video below may still have hands in other frames.")
    show_image(frame, f"Frame {test_index} (no hand detected)")

### 6b. Run on every frame

Now we loop over the whole video. The timestamp for frame *i* is `i × 1000 / fps` milliseconds.

We keep MediaPipe's result for every frame in a list called `detections`. Frames without a hand simply have an empty `hand_landmarks` list, so nothing crashes. We run the model **only once**; every later step (drawing, experimenting with brushes) reuses these results.

In [ ]:
def detect_video(path, size, fps, num_hands, min_detection, min_presence, min_tracking):
    """Run HandLandmarker on every frame. Returns one HandLandmarkerResult per frame."""
    detections = []
    cap = cv2.VideoCapture(path)
    start = time.time()
    with create_landmarker(num_hands, min_detection, min_presence, min_tracking) as landmarker:
        frame_index, last_timestamp = 0, -1
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            frame = cv2.resize(frame, size, interpolation=cv2.INTER_AREA)
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

            timestamp_ms = max(int(frame_index * 1000 / fps), last_timestamp + 1)  # must always increase
            result = landmarker.detect_for_video(mp_image, timestamp_ms)
            detections.append(result)

            last_timestamp = timestamp_ms
            frame_index += 1
            if frame_index % 100 == 0:
                print(f"  processed {frame_index} frames...")
    cap.release()

    found = sum(1 for r in detections if r.hand_landmarks)
    print(f"Done: {len(detections)} frames in {time.time() - start:.1f} s | "
          f"hand found in {found} frames ({100 * found / max(len(detections), 1):.0f}%)")
    if len(detections) == 0:
        warn("No frames could be read from the video.")
    elif found == 0:
        warn("No hand was detected anywhere in this video. Check the lighting and keep the whole "
             "hand in view, or lower the confidence settings in Section 10.")
    elif found < 0.5 * len(detections):
        warn("A hand was found in fewer than half of the frames, so the drawing will have gaps.")
    return detections


detections = detect_video(video_path, FRAME_SIZE, FPS, NUM_HANDS, MIN_HAND_DETECTION_CONFIDENCE,
                          MIN_HAND_PRESENCE_CONFIDENCE, MIN_TRACKING_CONFIDENCE)

---
## 7. Extract Finger Coordinates

To draw, we need **one point per frame**: the fingertip, in **pixels**.

- `hand_landmarks[0]` → the first (and only) hand
- `[8]` → landmark 8, the index fingertip
- `x * WIDTH`, `y * HEIGHT` → pixel coordinates

When no hand is found we return `None`. That tells the AirCanvas to lift the pen.

In [ ]:
TRACKED_LANDMARK = 8   # 4 = thumb tip, 8 = index tip, 12 = middle tip, 16 = ring tip, 20 = pinky tip


def to_pixel(landmark, width, height):
    """Normalized [0, 1] landmark -> integer pixel coordinates."""
    return int(landmark.x * width), int(landmark.y * height)


def get_fingertip(result, landmark_id, width, height, hand_index=0):
    """Pixel position of one landmark on one hand, or None if that hand wasn't detected."""
    if len(result.hand_landmarks) <= hand_index:
        return None
    landmark = result.hand_landmarks[hand_index][landmark_id]
    return to_pixel(landmark, width, height)


trajectory = [get_fingertip(r, TRACKED_LANDMARK, WIDTH, HEIGHT) for r in detections]

print(f"Tracking landmark {TRACKED_LANDMARK} ({LANDMARK_NAMES[TRACKED_LANDMARK]})")
print("First few positions:", [p for p in trajectory if p is not None][:5])

points = np.array([p for p in trajectory if p is not None])
if len(points):
    plt.figure(figsize=(6, 6 * HEIGHT / WIDTH))
    plt.scatter(points[:, 0], points[:, 1], c=np.arange(len(points)), cmap="viridis", s=8)
    plt.xlim(0, WIDTH)
    plt.ylim(HEIGHT, 0)   # image coordinates: y grows downward
    plt.colorbar(label="frame order")
    plt.title(f"Path of {LANDMARK_NAMES[TRACKED_LANDMARK]}")
    plt.gca().set_aspect("equal")
    plt.show()
else:
    warn("No fingertip positions to plot. No hand was detected.")

---
## 8. AirCanvas

The **AirCanvas** is an image that stays the same from frame to frame. Each frame we draw a line from the **previous** fingertip position to the **current** one.

Rules that keep the drawing clean:
- **Hand lost → lift the pen.** If a frame has no hand, we forget the previous point. The next detection starts a new stroke instead of drawing a long line across the canvas.
- **Smoothing (optional).** Landmarks jitter slightly from frame to frame. An *exponential moving average* blends each new point with the previous smoothed point: `smoothed = s * smoothed + (1 - s) * new`. `0` means no smoothing; around `0.5`–`0.7` gives smooth lines.

### Drawing modes
- **`"continuous"`**: draw whenever the hand is visible. Simple and good for the first run.
- **`"pinch"`**: draw only while the **thumb tip (4)** and **index tip (8)** are pinched together, like holding a pen. We compare the pinch distance to the hand's size (wrist → middle-finger knuckle), so it works whether the hand is near or far from the camera.

In [ ]:
class AirCanvas:
    def __init__(self, width, height, color=(255, 0, 0), thickness=6, background="white", smoothing=0.0):
        self.color = color              # BGR! (255, 0, 0) is blue
        self.thickness = thickness
        self.smoothing = smoothing
        bg_value = 255 if background == "white" else 0
        self.image = np.full((height, width, 3), bg_value, dtype=np.uint8)
        self.prev_point = None          # last point drawn (None = pen is up)
        self.smoothed = None            # running smoothed position (floats)

    def lift_pen(self):
        """End the current stroke so the next point doesn't connect to it."""
        self.prev_point = None
        self.smoothed = None

    def draw(self, point, color=None):
        """Extend the current stroke to `point` (or lift the pen if point is None)."""
        if point is None:
            self.lift_pen()
            return
        if self.smoothed is None or self.smoothing == 0:
            self.smoothed = np.array(point, dtype=float)
        else:
            self.smoothed = self.smoothing * self.smoothed + (1 - self.smoothing) * np.array(point)
        current = tuple(int(v) for v in self.smoothed)

        color = color or self.color
        if self.prev_point is None:
            cv2.circle(self.image, current, self.thickness // 2, color, -1, cv2.LINE_AA)
        else:
            cv2.line(self.image, self.prev_point, current, color, self.thickness, cv2.LINE_AA)
        self.prev_point = current


def pinch_ratio(hand_landmarks, width, height):
    """Thumb-index distance divided by hand size. Small value = pinching."""
    thumb, index = np.array(to_pixel(hand_landmarks[4], width, height)), np.array(to_pixel(hand_landmarks[8], width, height))
    wrist, knuckle = np.array(to_pixel(hand_landmarks[0], width, height)), np.array(to_pixel(hand_landmarks[9], width, height))
    hand_size = max(np.linalg.norm(wrist - knuckle), 1.0)
    return np.linalg.norm(thumb - index) / hand_size


def pen_is_down(result, mode, width, height, pinch_threshold):
    """Should we draw on this frame?"""
    if not result.hand_landmarks:
        return False
    if mode == "pinch":
        return pinch_ratio(result.hand_landmarks[0], width, height) < pinch_threshold
    return True   # "continuous"


def update_canvas(canvas, result, landmark_id, mode, pinch_threshold):
    """One frame of drawing. Returns (brush_point, pen_down) for visualization."""
    h, w = canvas.image.shape[:2]
    point = get_fingertip(result, landmark_id, w, h)
    pen_down = pen_is_down(result, mode, w, h, pinch_threshold)
    canvas.draw(point if pen_down else None)
    return point, pen_down


# Drawing settings (BGR colours)
BRUSH_COLOR = (255, 80, 0)     # blue-ish
BRUSH_THICKNESS = 6
BACKGROUND = "white"           # "white" or "black"
SMOOTHING = 0.5                # 0 = off, 0.5–0.7 = smooth, 0.9 = very laggy
DRAW_MODE = "continuous"       # "continuous" or "pinch"
PINCH_THRESHOLD = 0.35         # used only in "pinch" mode

# Quick preview: draw the whole trajectory onto a canvas (no video yet)
canvas = AirCanvas(WIDTH, HEIGHT, BRUSH_COLOR, BRUSH_THICKNESS, BACKGROUND, SMOOTHING)
for r in detections:
    update_canvas(canvas, r, TRACKED_LANDMARK, DRAW_MODE, PINCH_THRESHOLD)
show_image(canvas.image, f"AirCanvas preview ({DRAW_MODE} mode)", width=6)

**Pinch mode check:** the cell below prints how the pinch ratio changes through the video, which helps you choose `PINCH_THRESHOLD`. Values below the threshold count as "pen down".

In [ ]:
ratios = [pinch_ratio(r.hand_landmarks[0], WIDTH, HEIGHT) if r.hand_landmarks else np.nan for r in detections]
if np.all(np.isnan(ratios)):
    warn("No hand detected, so there are no pinch ratios to show.")
else:
    plt.figure(figsize=(10, 3))
    plt.plot(ratios, label="pinch ratio")
    plt.axhline(PINCH_THRESHOLD, color="red", linestyle="--", label=f"threshold = {PINCH_THRESHOLD}")
    plt.xlabel("frame")
    plt.legend()
    plt.title("Thumb–index distance / hand size")
    plt.show()
    pinched = np.nansum(np.array(ratios) < PINCH_THRESHOLD)
    print(f"Pinch detected in {pinched} of {len(ratios)} frames")

---
## 9. Generate Results

Now we go through the video one more time. For each frame we draw:
- **Output A**: the frame with the skeleton and a highlighted fingertip (🟢 green = pen down, ⚪ grey = pen up).
- **Output B**: the AirCanvas so far, with a small cursor showing where the finger is.
- **Output C**: A and B side by side.

All three are encoded as H.264 MP4 so they play right here in Colab.

In [ ]:
def put_title(image, text):
    cv2.putText(image, text, (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 0), 4, cv2.LINE_AA)
    cv2.putText(image, text, (10, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2, cv2.LINE_AA)
    return image


def render_outputs(path, detections, size, fps, landmark_id=8, color=(255, 80, 0), thickness=6,
                   background="white", smoothing=0.5, mode="continuous", pinch_threshold=0.35,
                   name="air_drawing"):
    width, height = size
    if not any(r.hand_landmarks for r in detections):
        warn("No hand was detected in this video, so the AirCanvas will be empty. "
             "Try a different video or lower the confidence settings.")

    canvas = AirCanvas(width, height, color, thickness, background, smoothing)
    out_tracking = Mp4Writer(os.path.join(OUTPUT_DIR, f"{name}_A_tracking.mp4"), fps, size)
    out_canvas = Mp4Writer(os.path.join(OUTPUT_DIR, f"{name}_B_canvas.mp4"), fps, size)
    out_combined = Mp4Writer(os.path.join(OUTPUT_DIR, f"{name}_C_combined.mp4"), fps, (width * 2, height))

    cap = cv2.VideoCapture(path)
    frames_with_pen_down = 0
    for result in detections:
        ok, frame = cap.read()
        if not ok:
            break
        frame = cv2.resize(frame, size, interpolation=cv2.INTER_AREA)

        point, pen_down = update_canvas(canvas, result, landmark_id, mode, pinch_threshold)
        frames_with_pen_down += pen_down
        cursor_color = (0, 200, 0) if pen_down else (160, 160, 160)

        # Output A: hand tracking
        tracking = frame.copy()
        if result.hand_landmarks:
            draw_hand(tracking, result.hand_landmarks[0], highlight_id=landmark_id,
                      highlight_color=cursor_color)
        else:
            put_title(tracking, "no hand")

        # Output B: canvas + live cursor (the cursor is not part of the drawing)
        canvas_view = canvas.image.copy()
        if point is not None:
            cv2.circle(canvas_view, point, thickness + 4, cursor_color, 2, cv2.LINE_AA)

        # Output C: side by side
        combined = np.hstack([put_title(tracking.copy(), "Hand Tracking"), put_title(canvas_view.copy(), "AirCanvas")])

        out_tracking.write(tracking)
        out_canvas.write(canvas_view)
        out_combined.write(combined)
    cap.release()

    if mode == "pinch" and frames_with_pen_down == 0 and any(r.hand_landmarks for r in detections):
        warn("Pinch mode never detected a pinch, so nothing was drawn. Try raising PINCH_THRESHOLD.")

    paths = {"tracking": out_tracking.close(), "canvas": out_canvas.close(), "combined": out_combined.close()}
    paths["drawing_png"] = os.path.join(OUTPUT_DIR, f"{name}_final_drawing.png")
    cv2.imwrite(paths["drawing_png"], canvas.image)
    return paths


outputs = render_outputs(video_path, detections, FRAME_SIZE, FPS, TRACKED_LANDMARK, BRUSH_COLOR,
                         BRUSH_THICKNESS, BACKGROUND, SMOOTHING, DRAW_MODE, PINCH_THRESHOLD)
print("Saved:", *outputs.values(), sep="\n  ")

### 🎉 The result

In [ ]:
show_video(outputs["combined"], width=min(2 * WIDTH, 960))

The individual outputs, plus download links. You can also find every file in the 📁 **Files** panel on the left, inside the `outputs` folder.

In [ ]:
show_video(outputs["tracking"], width=min(WIDTH, 420))
show_video(outputs["canvas"], width=min(WIDTH, 420))
display(HTML("<br>".join(download_link(outputs[k]) for k in ("tracking", "canvas", "combined"))))
show_image(cv2.imread(outputs["drawing_png"]), "Final drawing", width=6)

---
## 10. Experimentation

Change the values below and re-run the cells to see what happens. Ideas:

**Model settings** (Cell 10a re-runs MediaPipe, which takes a little longer)
- Set all confidences to `0.2`: does the hand get detected in more frames? Do you see false detections?
- Set them to `0.9`: what happens to the gaps in the drawing?

**Drawing settings** (Cell 10b only redraws, so it's fast)
- `TRACKED_LANDMARK = 4` (thumb) or `12` (middle finger)
- `SMOOTHING = 0` vs `0.8`: compare how jittery the lines are
- `DRAW_MODE = "pinch"`: draw only while pinching (use the pinch plot in Section 8 to pick a threshold)
- `BACKGROUND = "black"` with `BRUSH_COLOR = (0, 255, 255)` (yellow, in BGR) for a neon look

In [ ]:
# 10a. Model settings
NUM_HANDS = 1
MIN_HAND_DETECTION_CONFIDENCE = 0.5
MIN_HAND_PRESENCE_CONFIDENCE = 0.5
MIN_TRACKING_CONFIDENCE = 0.5

detections = detect_video(video_path, FRAME_SIZE, FPS, NUM_HANDS, MIN_HAND_DETECTION_CONFIDENCE,
                          MIN_HAND_PRESENCE_CONFIDENCE, MIN_TRACKING_CONFIDENCE)

In [ ]:
# 10b. Drawing settings
TRACKED_LANDMARK = 8
BRUSH_COLOR = (0, 255, 255)    # BGR
BRUSH_THICKNESS = 8
BACKGROUND = "black"
SMOOTHING = 0.6
DRAW_MODE = "continuous"
PINCH_THRESHOLD = 0.35

experiment = render_outputs(video_path, detections, FRAME_SIZE, FPS, TRACKED_LANDMARK, BRUSH_COLOR,
                            BRUSH_THICKNESS, BACKGROUND, SMOOTHING, DRAW_MODE, PINCH_THRESHOLD,
                            name="experiment")
show_video(experiment["combined"], width=min(2 * WIDTH, 960))
display(HTML(download_link(experiment["combined"])))

---
## 11. Optional Challenges

For those who finish early. Each challenge has a working solution below it. Try it yourself first!

### Challenge 1: 🌈 Rainbow brush
Make the brush colour change over time. *Hint:* build a colour in **HSV** with a hue that grows with the frame number, then convert it to BGR. `AirCanvas.draw()` accepts an optional `color`.

In [ ]:
def rainbow_color(frame_index, speed=3):
    hsv = np.uint8([[[(frame_index * speed) % 180, 255, 255]]])
    return tuple(int(c) for c in cv2.cvtColor(hsv, cv2.COLOR_HSV2BGR)[0, 0])


rainbow = AirCanvas(WIDTH, HEIGHT, thickness=10, background="black", smoothing=0.5)
for i, r in enumerate(detections):
    rainbow.draw(get_fingertip(r, TRACKED_LANDMARK, WIDTH, HEIGHT), color=rainbow_color(i))
show_image(rainbow.image, "Rainbow brush", width=6)

### Challenge 2: ✌️ Two hands, two colours
Detect up to two hands and give each its own colour. *Hint:* set `num_hands=2`. The order of hands in `hand_landmarks` can change between frames, so use `result.handedness[i][0].category_name` ("Left"/"Right") to decide which canvas stroke a hand belongs to.

> MediaPipe assumes a **mirrored** (selfie) image when labelling handedness, so on a normal recording "Left" and "Right" may look swapped. That's fine here; we only need the labels to stay consistent.

*(This needs a video with two hands visible to be interesting. With one hand it still works and draws in a single colour.)*

In [ ]:
two_hand_detections = detect_video(video_path, FRAME_SIZE, FPS, 2, MIN_HAND_DETECTION_CONFIDENCE,
                                   MIN_HAND_PRESENCE_CONFIDENCE, MIN_TRACKING_CONFIDENCE)

hand_colors = {"Left": (255, 80, 0), "Right": (0, 0, 255)}   # blue / red
image = np.full((HEIGHT, WIDTH, 3), 255, dtype=np.uint8)
strokes = {name: AirCanvas(WIDTH, HEIGHT, color, 6, smoothing=0.5) for name, color in hand_colors.items()}
for s in strokes.values():
    s.image = image                     # both hands draw on the same picture

for r in two_hand_detections:
    seen = set()
    for i, handedness in enumerate(r.handedness):
        hand_label = handedness[0].category_name
        if hand_label in strokes and hand_label not in seen:
            strokes[hand_label].draw(get_fingertip(r, TRACKED_LANDMARK, WIDTH, HEIGHT, hand_index=i))
            seen.add(hand_label)
    for hand_label in strokes.keys() - seen:   # that hand disappeared -> lift its pen
        strokes[hand_label].lift_pen()

show_image(image, "Two-hand drawing (blue = Left label, red = Right label)", width=6)

### Challenge 3: ✨ Augmented-reality ink
Instead of a separate canvas, paint the drawing **on top of the video** so the ink seems to float in the air. *Hint:* draw on a black canvas, then wherever the canvas isn't black, replace (or blend) the video pixel with the ink colour.

In [ ]:
ar_canvas = AirCanvas(WIDTH, HEIGHT, (0, 255, 255), 8, background="black", smoothing=0.5)
ar_writer = Mp4Writer(os.path.join(OUTPUT_DIR, "challenge_ar_ink.mp4"), FPS, FRAME_SIZE)
cap = cv2.VideoCapture(video_path)
for r in detections:
    ok, frame = cap.read()
    if not ok:
        break
    frame = cv2.resize(frame, FRAME_SIZE, interpolation=cv2.INTER_AREA)
    update_canvas(ar_canvas, r, TRACKED_LANDMARK, "continuous", PINCH_THRESHOLD)

    glow = cv2.GaussianBlur(ar_canvas.image, (0, 0), 6)          # soft glow around the ink
    frame = cv2.add(frame, glow)
    ink = ar_canvas.image.any(axis=2)
    frame[ink] = ar_canvas.image[ink]                             # solid ink on top
    ar_writer.write(frame)
cap.release()
show_video(ar_writer.close(), width=min(WIDTH, 640))

### More ideas (no solutions, so get creative)
- **Eraser gesture:** clear the canvas when the hand is held open with all five fingers up. *(Compare each fingertip's `y` to its PIP joint.)*
- **Brush size from depth:** use the fingertip's `z` value so the line gets thicker as the finger moves toward the camera.
- **Jump filter:** if the fingertip jumps more than ~15% of the frame width in one frame, treat it as a tracking glitch and lift the pen.
- **Finger counter:** show how many fingers are raised in the corner of Output A.

---
**Thanks for joining Camera, Code, Action!** 🎬 Built with [MediaPipe](https://ai.google.dev/edge/mediapipe/solutions/vision/hand_landmarker).